# 🌆 Smart City GIS Analytics — 강남구 보행성 분석

> OpenStreetMap + 보행성 지수 + 15분 도시 지수로 도시 정책 시뮬레이션.

**Research Question**: 강남구 어느 지역이 가장 보행 친화적인가? 15분 도시 기준으로 부족한 지역은?

In [ ]:
import sys; sys.path.insert(0, '../src')
import pandas as pd, numpy as np, geopandas as gpd
import matplotlib.pyplot as plt, contextily as ctx

from smartcity_gis import compute_walkability, AccessibilityAnalyzer
from smartcity_gis.api_clients import OSMClient, NSDIClient
from smartcity_gis.accessibility import fifteen_min_city_score

## 1. 도로망 + POI 수집

In [ ]:
# OSMnx로 강남구 보행 네트워크 + 편의시설
network = OSMClient.fetch_road_network('서울특별시 강남구', 'walk')
amenities = OSMClient.fetch_amenities(
    '서울특별시 강남구',
    tags={'amenity': ['restaurant', 'cafe', 'school', 'hospital']},
)
print(f'네트워크 노드: {len(network.nodes):,}')
print(f'편의시설: {len(amenities):,}')

## 2. 100m × 100m 그리드 생성

In [ ]:
boundary = NSDIClient().fetch_admin_boundary('sgg', sido_code='11')
gangnam = boundary[boundary.SIG_CD == '11680']

# fishnet 생성 (utility 함수)
from smartcity_gis.walkability import compute_walkability
# grid = create_fishnet(gangnam, cell_size=100)  # TODO
grid = None  # placeholder

## 3. 보행성 지수 계산

In [ ]:
walkability = compute_walkability(
    grid=grid,
    amenities=amenities,
    network=network,
    decay='exponential',
)

fig, ax = plt.subplots(figsize=(12, 12))
walkability.plot(column='walkability_score', cmap='YlGn', legend=True, ax=ax)
ctx.add_basemap(ax, crs=walkability.crs, source=ctx.providers.CartoDB.Positron)

## 4. 15분 도시 지수

In [ ]:
score_15min = fifteen_min_city_score(grid, amenities, network)

fig, ax = plt.subplots(figsize=(12, 12))
grid.assign(score=score_15min).plot(
    column='score', cmap='RdYlGn', vmin=0, vmax=6, legend=True, ax=ax
)
ax.set_title('강남구 15-Minute City Score (Moreno 2021)')

## 5. 등시선 분석

In [ ]:
analyzer = AccessibilityAnalyzer(network)
iso = analyzer.get_isochrone(
    origin=(127.0473, 37.5172),  # 강남역
    trip_times=[5, 10, 15, 20],
)

fig, ax = plt.subplots(figsize=(10, 10))
iso.isochrones.plot(column='minutes', cmap='viridis', alpha=0.5, legend=True, ax=ax)

## 6. 정책 시사점

- **Walkability 핫스팟**: 신사역·논현역 일대 (POI 다양성 高)
- **15분 도시 미흡 지역**: 개포동·일원동 (대단지 아파트 위주, 상업 부족)
- **권고**: 미흡 지역에 근린상업·의료·문화시설 확충 필요